# Data, Models, and Explanations

A compact orientation to the offline Breast Cancer Wisconsin benchmark. We compare prediction quality separately from model explanations.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, roc_auc_score
from xai_eval.data import breast_cancer_split
from xai_eval.models import make_models
from xai_eval.explainers import local_perturbation_attribution, model_specific_attribution

x_train, x_test, y_train, y_test, feature_names = breast_cancer_split(random_state=42)
baseline = np.median(x_train, axis=0)
rows = []
models = make_models(random_state=42)
for name, model in models.items():
    model.fit(x_train, y_train)
    rows.append({"model": name, "accuracy": accuracy_score(y_test, model.predict(x_test)),
                 "roc_auc": roc_auc_score(y_test, model.predict_proba(x_test)[:, 1])})
pd.DataFrame(rows)

In [ ]:
x = x_test[0]
model = models["LogisticRegression"]
local = local_perturbation_attribution(model, x, baseline=baseline)
model_based = model_specific_attribution(model, x)
top = np.argsort(-np.abs(local))[:8]
pd.DataFrame({"feature": np.asarray(feature_names)[top],
              "local_replacement": local[top],
              "linear_contribution": model_based[top]})